In [29]:
library(mlr)
library(tidyverse)
library(parallelMap)
library(parallel)

In [2]:
bestparams<-readRDS('best_hyperparameters.rds')
bestparams

$kernel
[1] "polynomial"

$degree
[1] 1

$cost
[1] 6.053465

$gamma
[1] 7.30625

In [9]:
data(spam, package = 'kernlab')
spamtib<-as_tibble(spam)
spamtask<-makeClassifTask(data = spamtib, target = 'type')

Warning message in makeTask(type = type, data = data, weights = weights, blocking = blocking, :
“Provided data is not a pure data.frame but from class tbl_df, hence it will be converted.”


In [4]:
tunedsvm<-setHyperPars(makeLearner('classif.svm'), par.vals = bestparams)

In [11]:
model<-train(tunedsvm,spamtask)

In [12]:
model

Model for learner.id=classif.svm; learner.class=classif.svm
Trained on: task.id = spamtib; obs = 4601; features = 57
Hyperparameters: kernel=polynomial,degree=1,cost=6.05,gamma=7.31

In [ ]:
cvfortuning<-makeResampleDesc('Holdout', split = 2/3)

#3 fold cross validation
outer<-makeResampleDesc('CV', iters=3)

randsearch<-makeTuneControlRandom(maxit = 100)

kernels<-c('polynomial','radial','sigmoid')

svmparamspace<-makeParamSet(
    makeDiscreteParam('kernel',values = kernels),
    makeIntegerParam('degree', lower = 1, upper = 3),
    makeNumericParam('cost', lower=0.1, upper = 10),
    makeNumericParam('gamma', lower=0.1, upper = 10)
)

svmwrapper<-makeTuneWrapper(
    'classif.svm', 
    resampling = cvfortuning, 
    par.set = svmparamspace,
    control = randsearch
)

svmwrapper

In [34]:
parallelStartSocket(cpus = detectCores())

resample(svmwrapper, spamtask, outer)

parallelStop()

Warning message in parallelStart(mode = MODE_SOCKET, cpus = cpus, socket.hosts = socket.hosts, :
“Parallelization was not stopped, doing it now.”
Stopped parallelization. All cleaned up.

Starting parallelization in mode=socket with cpus=16.

Exporting objects to slaves for mode socket: .mlr.slave.options

Resampling: cross-validation

Measures:             mmce      

Mapping in parallel: mode = socket; level = mlr.resample; cpus = 16; elements = 3.

